# TOKATÇI — B1 / B2 / B2C ayrı AI + policy

Üç ayrı kombinasyon:

- **B1 + policy_b1 + AI_b1**
- **B2 + policy_b2 + AI_b2**
- **B2C + policy_b2c + AI_b2c**

Veri: **777 Top2 episode**, sadece Petit Canard tarafı, 1.900 game, 202.630 ham karar; maxCount açılımı sonrası 209.179 satır.

Dataset farkı:
- B1: base game weight 1.0/0.5
- B2: turn<=3 setup x1.6, tech x0.35, retreat x0.6
- B2C: turn<=2 setup x1.2, tech x0.7, retreat x0.8

B1/B2/B2C şu an aynı decki kullanıyor: `deck_top2.csv` (Top2, `-1 Chi-Yu +1 Ultra Ball`).


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile, shutil

INPUT = Path('/kaggle/input')
print('INPUT entries:', [p.name for p in INPUT.iterdir()] if INPUT.exists() else 'MISSING')

def find_dir_with_file(filename, max_depth=3):
    for p in INPUT.rglob(filename):
        if p.is_file():
            return p.parent
    return None

DATA_DIR = find_dir_with_file('dataset_tokatci_b1.npz')
CODE = find_dir_with_file('src/mazlum/train_bc.py')
if CODE is None:
    # Some Kaggle datasets keep the zip unextracted.
    z = next((p for p in INPUT.rglob('tokatci_code.zip') if p.is_file()), None)
    if z is not None:
        dest = Path('/kaggle/working/tokatci_code')
        if dest.exists(): shutil.rmtree(dest)
        dest.mkdir(parents=True)
        with zipfile.ZipFile(z) as f: f.extractall(dest)
        CODE = dest
if DATA_DIR is None:
    raise FileNotFoundError('dataset_tokatci_b1.npz bulunamadı; tokatci_data datasetini Add Input yap.')
if CODE is None or not (CODE/'src/mazlum/train_bc.py').is_file():
    raise FileNotFoundError('src/mazlum/train_bc.py bulunamadı; tokatci_code.zip / tokatci-code input ekle.')

OUT = Path('/kaggle/working')
DECK = DATA_DIR/'deck_top2.csv'
DATASETS = {
    'b1':  DATA_DIR/'dataset_tokatci_b1.npz',
    'b2':  DATA_DIR/'dataset_tokatci_b2.npz',
    'b2c': DATA_DIR/'dataset_tokatci_b2c.npz',
}
print('DATA_DIR', DATA_DIR)
print('CODE', CODE)
for n,p in DATASETS.items(): print(n, p, p.is_file())


In [ ]:
variants = {
    'b1':  {'seed': 42, 'dropout': 0.10, 'lr': 1e-3},
    'b2':  {'seed': 43, 'dropout': 0.15, 'lr': 1e-3},
    'b2c': {'seed': 44, 'dropout': 0.10, 'lr': 1e-3},
}
for name, cfg in variants.items():
    out = OUT / f'model_tokatci_{name}.npz'
    cmd = [
        sys.executable, str(CODE/'src/mazlum/train_bc.py'),
        '--dataset', str(DATASETS[name]), '--out', str(out),
        '--device', 'cuda', '--batch-size', '256',
        '--epochs', '30', '--patience', '5',
        '--dropout', str(cfg['dropout']), '--lr', str(cfg['lr']),
        '--arena-every', '0', '--select-metric', 'val',
        '--print-every', '1', '--seed', str(cfg['seed']),
    ]
    print('\n===', name, '===')
    subprocess.run(cmd, check=True)


In [ ]:
import numpy as np, torch
sys.path.insert(0, str(CODE/'src/mazlum'))
import model as M

def load_model(path):
    d=np.load(path); net=M.MazlumNet()
    net.load_state_dict({k:torch.from_numpy(d[k].astype(np.float32)) for k in d.files if k not in ('state_mean','state_std','option_mean','option_std')})
    net.eval()
    norm={k:torch.from_numpy(d[k].astype(np.float32)) for k in ('state_mean','state_std','option_mean','option_std')}
    return net,norm

@torch.no_grad()
def predict(net,norm,data,idx):
    out=[]
    for start in range(0,len(idx),2048):
        ids=idx[start:start+2048]
        s=torch.from_numpy(data['state'][ids].astype(np.float32)); o=torch.from_numpy(data['options'][ids].astype(np.float32))
        s=(s-norm['state_mean'])/norm['state_std']; o=(o-norm['option_mean'])/norm['option_std']
        logits=net(s,o,torch.from_numpy(data['card_ids'][ids].astype(np.int64)),torch.from_numpy(data['mask'][ids].astype(np.float32)))
        out.append(logits.argmax(-1).numpy())
    return np.concatenate(out)

for name,path in DATASETS.items():
    d=np.load(path); val=np.nonzero(d['split']==1)[0]; y=d['label'][val]; train=np.nonzero(d['split']==0)[0]
    maj=int(np.bincount(d['label'][train]).argmax()); st=d['selected_type'][val]; ctx=d['context'][val]; act=d['selected_actor'][val]
    net,norm=load_model(OUT/f'model_tokatci_{name}.npz'); p=predict(net,norm,d,val)
    print('\n',name,'val',len(y),'acc',(p==y).mean(),'majority',(y==maj).mean())
    for k,m in {'PLAY':st==7,'ATTACK':st==13,'ATTACH':st==8,'ABILITY':st==10,
                'TARGET':np.isin(ctx,[3,4,5,13,14,16,22,25,40]),
                'PLAY_RISKY_1260':(st==7)&(act==1260),
                'ABILITY_MUNKI_112':(st==10)&(act==112)}.items():
        if m.any(): print(f'  {k:18} n={int(m.sum()):5d} acc={(p[m]==y[m]).mean():.4f} maj={(y[m]==maj).mean():.4f}')


## Policy katmanı

Üç AI ayrı eğitilir; inference'ta her birinin kendi policy wrapper'ı vardır:

- **B1:** `policy_b1` — saf BC, rule prior yok.
- **B2:** `policy_b2` — güçlü early-setup prior; tech cezası; exception'lı.
- **B2C:** `policy_b2c` — yumuşak prior; hard mask yok.

Ortak formül:

```text
final = model_logit + lambda * rule_prior
```

Search yalnızca düşük margin / kritik karar / prior-model çelişkisinde çalışır. Lethal varsa direkt oynanır.


## Lokal smoke

Her varyantı ayrı süreçlerde, farklı seed'lerle çalıştırın. Lokal sonuç eleme değil; illegal/crash/timeout kontrolü.


In [ ]:
# 4 process x 25 BO3 örneği (shell'de çalıştırmak daha güvenli):
# OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 python tools/arena.py --bot0 bots/tokatci_b1 --bot1 bots/tokatci_b2 --games 25 --bo3 --seed 101
